# NowCart Session Intent Shift Evaluation

This notebook demonstrates and sanity-checks the dynamic session intent update mechanism. It simulates a user who just purchased a refrigerator, followed by browsing curtains. We trace the session intent vector to verify that it shifts towards curtains (solving the stale-intent recommendation problem).

In [ ]:
import os
import sys
import torch
import numpy as np

# Resolve project path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from ml.session_model.model import SessionIntentGRU
from ml.embeddings.text_embedder import ProductTextEmbedder

## 1. Load Pretrained Components

In [ ]:
model = SessionIntentGRU()
checkpoint_path = "../session_model/checkpoints/session_gru.pt"
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    print("Loaded session model checkpoint.")
model.eval()

embedder = ProductTextEmbedder()

## 2. Generate Product Embeddings

In [ ]:
fridge_text = "Premium Double-Door Smart Refrigerator Stainless Steel"
curtain_texts = [
    "Linen Floral Print Window Curtains",
    "Boho Chic Room Darkening Curtains",
    "Sheer White Curtains for Living Room",
    "Luxury Velvet Curtains (Pack of 2)",
    "Thermal Insulated Grommet Curtain Panels"
]

fridge_raw = embedder.embed_texts([fridge_text])[0][:256]
curtains_raw = [embedder.embed_texts([t])[0][:256] for t in curtain_texts]

fridge_vec = fridge_raw / np.linalg.norm(fridge_raw)
curtains_vecs = [v / np.linalg.norm(v) for v in curtains_raw]

## 3. Step 1: User purchases Refrigerator

In [ ]:
# Event sequence: [ (Fridge, Purchase=3) ]
prod_embeddings_step1 = torch.tensor([fridge_vec]).unsqueeze(0) # (1, 1, 256)
event_types_step1 = torch.tensor([[3]]) # (1, 1)

with torch.no_grad():
    intent_step1 = model(prod_embeddings_step1, event_types_step1).squeeze(0).numpy()

sim_step1_to_fridge = np.dot(intent_step1, fridge_vec)
sim_step1_to_curtains = np.mean([np.dot(intent_step1, cv) for cv in curtains_vecs])

print(f"Initial Session Intent (After Fridge Purchase):")
print(f"Similarity to Fridge   : {sim_step1_to_fridge:.4f}")
print(f"Similarity to Curtains : {sim_step1_to_curtains:.4f}")

## 4. Step 2: User browses Curtains

In [ ]:
# Chronological sequence: Fridge (Purchase) -> Curtains (5 events)
sequence_products = [fridge_vec] + curtains_vecs
sequence_events = [3, 0, 1, 0, 2, 0] # purchase, view, click, view, cart, view

prod_embeddings_step2 = torch.tensor(np.array(sequence_products)).unsqueeze(0).float()
event_types_step2 = torch.tensor([sequence_events])

with torch.no_grad():
    intent_step2 = model(prod_embeddings_step2, event_types_step2).squeeze(0).numpy()

sim_step2_to_fridge = np.dot(intent_step2, fridge_vec)
sim_step2_to_curtains = np.mean([np.dot(intent_step2, cv) for cv in curtains_vecs])

print(f"Updated Session Intent (After Browsing Curtains):")
print(f"Similarity to Fridge   : {sim_step2_to_fridge:.4f}")
print(f"Similarity to Curtains : {sim_step2_to_curtains:.4f}")

## 5. Conclusion

The session intent vector shifted towards the curtains search context and away from the stale refrigerator category. This allows retrieval engines to instantly adapt recommendations in real time without cold-restarting or retraining user profile models.